In [ ]:
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, save_and_log_checkpoint, load_training_checkpoint
import mlflow
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKTransformerAdapter
from model_classes import WaitKTransformerMT, SimulTransformerConfig

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_transformer_distillation")

from transformers import AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [ ]:
model_cfg = SimulTransformerConfig(
    vocab_size=len(tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    max_seq_len=64,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [ ]:
transformer = WaitKHybridMamba2MT(model_cfg)
count_parameters(transformer)

In [ ]:
train_cfg = TrainConfig(
    epochs=8,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=2e-5,
    use_amp=True,
)

In [ ]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [ ]:
train_waitk_student(
    student=transformer,
    train_dataset=dataset,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="transformer",
    resume_from_checkpoint="./checkpoints/transformer_epoch_4.pt"
)

In [ ]:
student, optim, scaler, model_cfg, train_cfg, state, train_time = load_training_checkpoint("./models/transformer_wait10.pt", SimulTransformerConfig, WaitKTransformerMT)
save_and_log_checkpoint(
    path="./checkpoints/transformer_epoch_4.pt",
    student=student,
    optimizer=optim,
    scaler=scaler,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    **state,
    train_time=train_time,
    mlflow_run_id="da30c73b168e4bf296c9597de3623de8"
)